# 08 - ONNX export and size reduction without accuracy loss

`scenario_v1` is exported to ONNX with the temperature scaling and softmax **inside** the graph: the app feeds a 256x256 image and gets 5 probabilities that sum to 1.

Variants compared against the PyTorch model:

| variant | what is stored / computed |
|---|---|
| `fp32` | plain export, 32-bit weights and arithmetic |
| `int8_static` | full int8: weights **and** activations rounded to 8 bits (onnxruntime static QDQ, per-channel, calibrated on 300 train images) |
| `int8_weights` | weights stored as per-channel int8, small layers (< 50k weights) as fp16; converted back to fp32 when the model loads, so the arithmetic is fp32 |
| `fp16_weights` | weights stored as fp16; arithmetic fp32 |

Every variant is scored on validation, test and the mixed-stress holdout under all stress conditions, with the thresholds from notebook 07 left unchanged.

**Selection rule (fixed before looking at test):** the smallest file whose top-1 class matches PyTorch on at least 99.9% of validation images and whose status matches on at least 99%.

In [1]:
import sys
from pathlib import Path

ML = Path.cwd()
while not (ML / "src").exists():
    ML = ML.parent
sys.path.insert(0, str(ML))

import json
import shutil

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader

from src import calibrate as C
from src import export as E
from src.config import CACHE, SPLITS
from src.model import load_checkpoint
from src.stress import CONDITIONS, AllConditionsDataset
from src.train import CHECKPOINTS

ARTIFACTS = ML / "artifacts"
ONNX_DIR = CACHE / "onnx"
ONNX_DIR.mkdir(parents=True, exist_ok=True)
thresholds = json.loads((ARTIFACTS / "thresholds.json").read_text())
T, MIN_CONF, MULTI_CONF = thresholds["temperature"], thresholds["min_conf"], thresholds["multi_conf"]
model, ckpt = load_checkpoint(CHECKPOINTS / f"{thresholds['run']}_best.pt", "cpu")
reference = E.Calibrated(model, T).eval()
thresholds

{'run': 'scenario_v1',
 'temperature': 1.2379,
 'min_conf': 0.84,
 'multi_conf': 0.2,
 'target_confident_accuracy': 0.95,
 'fit_on': 'val split, seen stress conditions pooled',
 'classes': ['healthy', 'leaf_miner', 'rust', 'phoma', 'cercospora'],
 'rule': ['confident if top1 >= min_conf',
  'possibly_multiple if top1 and top2 are diseases, top2 >= multi_conf and top1 + top2 >= min_conf',
  'otherwise not_sure']}

## Build the variants

In [2]:
paths = {"fp32": E.export_fp32(model, T, ONNX_DIR / "fp32.onnx")}
paths["int8_static"] = E.quantize_int8(paths["fp32"], ONNX_DIR / "int8_static.onnx", E.calibration_images(300))
paths["int8_weights"] = E.compress_weights(paths["fp32"], ONNX_DIR / "int8_weights.onnx", "int8", int8_min_size=50_000)
paths["fp16_weights"] = E.compress_weights(paths["fp32"], ONNX_DIR / "fp16_weights.onnx", "fp16")

sizes = pd.DataFrame({name: {"size_MB": p.stat().st_size / 1e6, "cpu_latency_ms_1_thread": E.latency_ms(E.session(p))}
                      for name, p in paths.items()}).T
sizes.loc["pytorch checkpoint", "size_MB"] = (CHECKPOINTS / f"{thresholds['run']}_best.pt").stat().st_size / 1e6
sizes.round(2)

,size_MB,cpu_latency_ms_1_thread
fp32,6.10,8.31
int8_static,1.87,3.30
int8_weights,1.92,8.45
fp16_weights,3.09,8.37
pytorch checkpoint,6.22,NaN


Latency is a single 256x256 image on one CPU thread of this laptop; a budget phone will be several times slower, but the ratios between variants hold roughly. The weight-only variants run at fp32 speed because the arithmetic is fp32.

## Score every variant on every split and condition

In [3]:
sessions = {name: E.session(p, threads=4) for name, p in paths.items()}

def score_split(split):
    probs = {"pytorch": [], **{n: [] for n in sessions}}
    ds = AllConditionsDataset(SPLITS / f"{split}.csv", ckpt["config"]["size"])
    labels, cond = [], []
    with torch.no_grad():
        for x, y, k in DataLoader(ds, batch_size=64, num_workers=6):
            probs["pytorch"].append(reference(x).numpy())
            for n, s in sessions.items():
                probs[n].append(E.run(s, x.numpy()))
            labels.append(y.numpy())
            cond.append(k.numpy())
    df = pd.concat([ds.parts[0].df] * len(CONDITIONS), ignore_index=True)
    df["condition"] = [list(CONDITIONS)[k] for k in np.concatenate(cond)]
    df["seen"] = df["condition"].map(lambda c: CONDITIONS[c][0])
    return {n: np.concatenate(v) for n, v in probs.items()}, df

scored = {split: score_split(split) for split in ["val", "test", "mixed_holdout"]}

In [4]:
def parity(split):
    probs, _ = scored[split]
    ref, st_ref = probs["pytorch"], C.status(probs["pytorch"], MIN_CONF, MULTI_CONF)
    return pd.DataFrame({n: {"top1_agreement": np.mean(p.argmax(1) == ref.argmax(1)),
                             "status_agreement": np.mean(C.status(p, MIN_CONF, MULTI_CONF) == st_ref),
                             "mean_abs_prob_diff": np.abs(p - ref).mean(),
                             "max_abs_prob_diff": np.abs(p - ref).max(),
                             "max_sum_error": np.abs(p.sum(1) - 1).max()}
                         for n, p in probs.items() if n != "pytorch"}).T

parity_val = parity("val")
parity_val.style.format("{:.5f}")

,top1_agreement,status_agreement,mean_abs_prob_diff,max_abs_prob_diff,max_sum_error
fp32,1.00000,1.00000,0.00000,0.00001,0.00000
int8_static,0.71235,0.46857,0.11853,0.93131,0.01176
int8_weights,0.99910,0.98824,0.00111,0.02796,0.00000
fp16_weights,1.00000,0.99910,0.00024,0.01002,0.00000


In [5]:
def quality(split):
    probs, df = scored[split]
    y, seen = df["label"].to_numpy(), df["seen"].to_numpy()
    rows = {}
    for n, p in probs.items():
        pred, st = p.argmax(1), C.status(p, MIN_CONF, MULTI_CONF)
        conf = st == "confident"
        rows[n] = {"acc": np.mean(pred == y), "macro_f1": f1_score(y, pred, average="macro"),
                   "acc_when_confident (seen)": np.mean((pred == y)[conf & seen]),
                   "confident (seen)": conf[seen].mean(), "confident_wrong": np.mean(conf & (pred != y))}
    return pd.DataFrame(rows).T

print("Validation, all conditions pooled")
quality("val").style.format("{:.4f}")

Validation, all conditions pooled


,acc,macro_f1,acc_when_confident (seen),confident (seen),confident_wrong
pytorch,0.8218,0.7928,0.9503,0.5877,0.0299
fp32,0.8218,0.7928,0.9503,0.5877,0.0299
int8_static,0.6744,0.6199,0.9443,0.2680,0.0176
int8_weights,0.8209,0.7917,0.9514,0.5889,0.0294
fp16_weights,0.8218,0.7928,0.9503,0.5877,0.0299


## Pick the variant

In [6]:
ok = parity_val[(parity_val["top1_agreement"] >= 0.999) & (parity_val["status_agreement"] >= 0.99)]
chosen = min(ok.index, key=lambda n: paths[n].stat().st_size)
print(f"Chosen: {chosen} ({paths[chosen].stat().st_size / 1e6:.2f} MB)")
print("Passing variants:", list(ok.index))

Chosen: fp16_weights (3.09 MB)
Passing variants: ['fp32', 'fp16_weights']


## Final check on test and the mixed holdout (scored once)

In [7]:
print("Test, all conditions pooled")
display(quality("test").style.format("{:.4f}"))
print("Test parity vs PyTorch")
display(parity("test").style.format("{:.5f}"))

Test, all conditions pooled


,acc,macro_f1,acc_when_confident (seen),confident (seen),confident_wrong
pytorch,0.8258,0.7986,0.9720,0.5755,0.0153
fp32,0.8258,0.7986,0.9720,0.5755,0.0153
int8_static,0.6818,0.6264,0.9458,0.2853,0.0171
int8_weights,0.8272,0.7995,0.9731,0.5743,0.0149
fp16_weights,0.8263,0.7991,0.9720,0.5755,0.0153


Test parity vs PyTorch


,top1_agreement,status_agreement,mean_abs_prob_diff,max_abs_prob_diff,max_sum_error
fp32,1.00000,1.00000,0.00000,0.00001,0.00000
int8_static,0.70342,0.48065,0.11650,0.92718,0.01176
int8_weights,0.99685,0.99145,0.00108,0.01698,0.00000
fp16_weights,0.99955,0.99910,0.00024,0.01044,0.00000


In [8]:
probs, df = scored["test"]
rows = []
for cond in CONDITIONS:
    m = (df["condition"] == cond).to_numpy()
    for n in ["pytorch", chosen]:
        rows.append({"condition": cond, "model": n, **C.summarise(probs[n][m], df[m], MIN_CONF, MULTI_CONF)})
by_cond = pd.DataFrame(rows).pivot(index="condition", columns="model", values=["acc", "macro_f1", "acc_when_confident", "not_sure"]).loc[list(CONDITIONS)]
by_cond.round(3)

acc             macro_f1          \
model                   fp16_weights pytorch fp16_weights pytorch   
condition                                                           
clean                          0.871   0.871        0.845   0.845   
home background                0.837   0.832        0.804   0.797   
window side light              0.866   0.866        0.846   0.846   
dim warm lamp                  0.698   0.698        0.647   0.647   
picked leaf curl               0.856   0.856        0.833   0.833   
budget camera                  0.837   0.837        0.808   0.808   
handheld blur                  0.871   0.871        0.851   0.851   
realistic weekend photo        0.777   0.777        0.750   0.750   
UNSEEN cool LED light          0.851   0.851        0.838   0.838   
UNSEEN forwarded photo         0.762   0.762        0.737   0.737   
UNSEEN strong blur             0.861   0.861        0.850   0.850   

                        acc_when_confident             not_sure          
model                         fp16_weights pytorch fp16_weights pytorch  
condition                                                                
clean                                0.977   0.977        0.178   0.178  
home background                      0.975   0.975        0.233   0.233  
window side light                    0.985   0.985        0.193   0.193  
dim warm lamp                        0.910   0.910        0.396   0.396  
picked leaf curl                     0.992   0.992        0.198   0.198  
budget camera                        0.991   0.991        0.233   0.233  
handheld blur                        0.977   0.977        0.188   0.188  
realistic weekend photo              0.942   0.942        0.376   0.376  
UNSEEN cool LED light                0.973   0.973        0.277   0.277  
UNSEEN forwarded photo               0.959   0.959        0.272   0.267  
UNSEEN strong blur                   0.992   0.992        0.188   0.193

In [9]:
probs, df = scored["mixed_holdout"]
pd.DataFrame({n: {"flagged (not confident)": np.mean(C.status(p, MIN_CONF, MULTI_CONF) != "confident"),
                  "top1 agreement vs pytorch": np.mean(p.argmax(1) == probs["pytorch"].argmax(1))}
              for n, p in probs.items()}).T.round(4)

,flagged (not confident),top1 agreement vs pytorch
pytorch,0.8659,1.0000
fp32,0.8659,1.0000
int8_static,0.9368,0.4700
int8_weights,0.8613,0.9923
fp16_weights,0.8644,1.0000


## Write the export bundle

In [10]:
E.EXPORT.mkdir(exist_ok=True)
shutil.copy(paths[chosen], E.EXPORT / "leaf_model.onnx")
report = {
    "chosen_variant": chosen, "selection_rule": "smallest with val top-1 agreement >= 0.999 and status agreement >= 0.99",
    "sizes_MB": sizes["size_MB"].round(3).to_dict(),
    "cpu_latency_ms_1_thread_laptop": sizes["cpu_latency_ms_1_thread"].dropna().round(2).to_dict(),
    "parity_val": parity_val.round(6).to_dict(orient="index"),
    "parity_test": parity("test").round(6).to_dict(orient="index"),
    "quality_test": quality("test").round(4).to_dict(orient="index"),
}
E.write_bundle(E.EXPORT / "leaf_model.onnx", thresholds, report)

sess = E.session(E.EXPORT / "leaf_model.onnx")
sample = AllConditionsDataset(SPLITS / "val.csv", 256)[0][0][None].numpy()
out = sess.run(["probs"], {"image": sample})[0]
print("bundle:", sorted(p.name for p in E.EXPORT.iterdir()))
print("sample output:", out.round(4), "sum", out.sum().round(6))

bundle: ['export_report.json', 'labels.json', 'leaf_model.onnx', 'preprocess.json', 'thresholds.json']
sample output: [[0.4383 0.1023 0.1528 0.0208 0.2859]] sum 1.0
